# Building a Deep Neural Network from Scratch
**Objective:** To understand the mathematical engine behind Deep Learning by building a 2-layer Neural Network using pure Python and NumPy. No frameworks (like PyTorch or TensorFlow) were used.

This notebook covers:
1. Matrix multiplication (Forward Pass)
2. Activation functions (Sigmoid)
3. Log Loss calculation
4. The Chain Rule / Backpropagation 
5. Gradient Descent optimization

In [1]:
import numpy as np

X = np.array([[2.0, 50.0]])
W1 = np.array([[0.5, 0.2, -0.1], [0.1, -0.5, 0.3]])

b1 = np.array([[-2.0, 1.0, 0.0]])
Z1 = np.dot(X, W1) + b1

print("The raw hidden layer output (Z1) is:")
print(Z1)

The raw hidden layer output (Z1) is:
[[  4.  -23.6  14.8]]


In [2]:
def sigmoid(z):
    return 1/(1 + np.exp(-z))

A1 = sigmoid(Z1)

print("The final activated signals leaving the hidden layer (A1):")
print(np.round(A1, 4))

The final activated signals leaving the hidden layer (A1):
[[0.982 0.    1.   ]]


In [3]:
W2 = np.array([[0.8],
               [-0.4],
               [0.6]])

b2 = np.array([[-0.5]])

In [4]:
Z2 = np.dot(A1, W2) + b2
#print(Z2)
A2 = sigmoid(Z2)
print(A2)

[[0.70798357]]


In [5]:
Y = 0
dZ2 = A2 - Y

dW2 = np.dot(A1.T, dZ2)
db2 = np.sum(dZ2, axis=0, keepdims=True)

print("Weight Gradients for Layer 2 (dW2):")
print(dW2)
print(dZ2)

Weight Gradients for Layer 2 (dW2):
[[6.95249627e-01]
 [3.98724943e-11]
 [7.07983303e-01]]
[[0.70798357]]


In [6]:
sigmoid_derivative = A1 * (1 - A1)
dZ1 = np.dot(dZ2, W2.T) * sigmoid_derivative

dW1 = np.dot(X.T, dZ1)

db1 = np.sum(dZ1, axis=0, keepdims=True)

print("Layer 1 Blame Calculated!")

Layer 1 Blame Calculated!


In [7]:
# Set the speed limit (Learning Rate)
alpha = 0.01

# Turn the knobs! (Subtract a tiny fraction of the blame from the weights)
W2 = W2 - (alpha * dW2)
b2 = b2 - (alpha * db2)

W1 = W1 - (alpha * dW1)
b1 = b1 - (alpha * db1)

print("Weights successfully updated! The AI just got slightly smarter.")

Weights successfully updated! The AI just got slightly smarter.


### The Training Engine (1000 Epochs)

In [8]:
import numpy as np

# 1. The Patient Data (X) and the True Target (Y)
X = np.array([[2.0, 50.0]]) # Size: 2.0, Age: 50
Y = np.array([[0]])         # The truth: 0 (Benign/Safe)

# 2. Setup: Random starting weights and biases
np.random.seed(42) # Ensures we get the exact same random starting numbers
W1 = np.random.randn(2, 3) 
b1 = np.zeros((1, 3))
W2 = np.random.randn(3, 1)
b2 = np.zeros((1, 1))

# The Speed Limit
alpha = 0.05 

# The Squishifier
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("Starting Training Engine...\n")

# 3. THE TRAINING LOOP (Run 1000 times)
for epoch in range(1000):
    
    # --- STEP 1: FORWARD PASS (Shoot the arrow) ---
    Z1 = np.dot(X, W1) + b1
    A1 = sigmoid(Z1)
    
    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2)
    
    # --- STEP 2: CALCULATE LOSS (Measure the miss) ---
    # Using the simplified Log Loss for Y=0. 
    # (Adding + 1e-8 prevents the math from crashing if A2 becomes exactly 1.0)
    loss = -np.log(1 - A2 + 1e-8) 
    
    # Print progress every 200 cycles
    if epoch % 200 == 0:
        print(f"Cycle {epoch:03} | Loss: {loss[0][0]:.4f} | AI Guess: {A2[0][0]*100:.2f}% chance of cancer")
        
    # --- STEP 3: BACKWARD PASS (Assign the blame) ---
    # Output Layer Blame
    dZ2 = A2 - Y
    dW2 = np.dot(A1.T, dZ2)
    db2 = np.sum(dZ2, axis=0, keepdims=True)
    
    # Hidden Layer Blame (Break through the shield)
    sigmoid_derivative = A1 * (1 - A1)
    dZ1 = np.dot(dZ2, W2.T) * sigmoid_derivative
    dW1 = np.dot(X.T, dZ1)
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    
    # --- STEP 4: UPDATE WEIGHTS (Turn the knobs) ---
    W2 = W2 - (alpha * dW2)
    b2 = b2 - (alpha * db2)
    W1 = W1 - (alpha * dW1)
    b1 = b1 - (alpha * db1)

# The Final Result
print(f"\nTraining Complete!")
print(f"Final AI Prediction: {A2[0][0]*100:.6f}%")

Starting Training Engine...

Cycle 000 | Loss: 1.7666 | AI Guess: 82.91% chance of cancer
Cycle 200 | Loss: 0.0607 | AI Guess: 5.89% chance of cancer
Cycle 400 | Loss: 0.0280 | AI Guess: 2.76% chance of cancer
Cycle 600 | Loss: 0.0181 | AI Guess: 1.79% chance of cancer
Cycle 800 | Loss: 0.0133 | AI Guess: 1.32% chance of cancer

Training Complete!
Final AI Prediction: 1.050415%


In [9]:
def predict_cancer_risk(patient_data):
    # The Forward Pass (Inference)
    Z1_test = np.dot(patient_data, W1) + b1
    A1_test = sigmoid(Z1_test)

    Z2_test = np.dot(A1_test, W2) + b2
    A2_test = sigmoid(Z2_test)

    # Extract the actual number from the 1x1 matrix
    final_percentage = A2_test[0][0] 
    
    return final_percentage

### Inference & The Overfitting Phenomenon

In [10]:
# Patient 1: The healthy patient we trained on (Size: 2.0, Age: 50)
patient_1 = np.array([[2.0, 50.0]])

# Patient 2: The high-risk patient (Size: 8.5, Age: 65)
patient_2 = np.array([[8.5, 65.0]])

# Patient 3: A young patient with a tiny, suspicious bump (Size: 1.0, Age: 25)
patient_3 = np.array([[1.0, 25.0]])

# Run the diagnoses!
print(f"Patient 1 Risk: {predict_cancer_risk(patient_1) * 100:.2f}%")
print(f"Patient 2 Risk: {predict_cancer_risk(patient_2) * 100:.2f}%")
print(f"Patient 3 Risk: {predict_cancer_risk(patient_3) * 100:.2f}%")

Patient 1 Risk: 1.05%
Patient 2 Risk: 1.05%
Patient 3 Risk: 1.05%
